# Treino Grupo B (QR) — Kaggle T4

**Autor**: Massanori  
**Data**: 19/05/2026  
**Descrição**: Orquestra smoke + treino completo + sanity check Pearson do Grupo B (`QuantileRegressionModule` com `qr_loss`, alpha=0.10) no S5 do TCC. Réplica de Giannakopoulos et al. (2026, seção III).

## Pré-requisitos

- Settings → Accelerator = **T4 x1** (ou P100)
- Add Input → **tcc-mri-recons-varnet-brain-4x**

## Fluxo

1. Célula 2: clone do repo + instala dependencias
2. Célula 3: **diagnóstico** (GPU + estrutura do dataset)
3. Célula 4: **smoke 500 iters** (gate antes do treino full, ~5 min em T4)
4. Célula 5: **treino completo 210k iters** (~6-12h GPU)
5. Célula 6: **Pearson sanity check** — esperado r ~ 0,91 (paper Tabela 1)
6. Célula 7: empacota outputs
7. Célula 8: publica como Kaggle dataset `tcc-mri-qr-checkpoints` via API

## Diferenças em relação ao Grupo A

- Módulo: `ResidualMagnitudeModule` → `QuantileRegressionModule` (~15.5M params)
- Loss: `resm_loss` → `qr_loss` com `alpha=0.10`
- Scripts: `train_resm.py` → `train_qr.py`
- Mesma seed (42) e hiperparametros (controle experimental D4)

In [ ]:
# Célula 2 — Setup: clone repo + dependencias
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/KR0N0S7/tcc-mri-uncertainty.git'
REPO_DIR = Path('/kaggle/working/tcc-mri-uncertainty')

if REPO_DIR.is_dir():
    print(f'Repo ja existe em {REPO_DIR}; git pull...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('\nHEAD do repo:')
subprocess.run(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'], check=False)

print('\nInstalando dependencias...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(REPO_DIR / 'requirements.txt'),
    'python-dotenv',
], check=True)
print('Setup OK')

In [ ]:
# Célula 3 — Diagnóstico OBRIGATÓRIO. NAO prossiga em caso de falha.
import os
import subprocess
import sys
from pathlib import Path

DATASET_SLUG = 'tcc-mri-recons-varnet-brain-4x'
EXPECTED_SUBS = {'train', 'val', 'cal', 'test'}


def find_dataset_root(slug):
    candidates = [Path('/kaggle/input') / slug]
    datasets_root = Path('/kaggle/input/datasets')
    if datasets_root.is_dir():
        for user_dir in datasets_root.iterdir():
            if user_dir.is_dir():
                candidates.append(user_dir / slug)
    print('Candidatos a investigar:')
    for c in candidates:
        print(f'  {c} (existe: {c.is_dir()})')
    for cand in candidates:
        if not cand.is_dir():
            continue
        subs = {p.name for p in cand.iterdir() if p.is_dir()}
        if EXPECTED_SUBS.issubset(subs):
            return cand
        sub_dirs = [p for p in cand.iterdir() if p.is_dir()]
        if len(sub_dirs) == 1 and (sub_dirs[0] / 'train').is_dir():
            print(f'  Aninhamento extra detectado em {cand}; usando {sub_dirs[0]}')
            return sub_dirs[0]
    raise FileNotFoundError(f'Dataset "{slug}" nao encontrado.')


print('=' * 60)
print('DIAGNOSTICO')
print('=' * 60)

print('\n[1] Conteudo de /kaggle/input/:')
subprocess.run(['ls', '-la', '/kaggle/input/'], check=False)
if Path('/kaggle/input/datasets').is_dir():
    print('\n    Conteudo de /kaggle/input/datasets/:')
    subprocess.run(['ls', '-la', '/kaggle/input/datasets/'], check=False)

print(f'\n[2] Procurando dataset com slug "{DATASET_SLUG}":')
actual_root = find_dataset_root(DATASET_SLUG)
print(f'\n  -> RECONS_ROOT resolvido: {actual_root}')

print('\n[3] Subpastas do dataset:')
for sub in ('train', 'val', 'cal', 'test'):
    p = actual_root / sub
    if not p.is_dir():
        raise FileNotFoundError(f'Subpasta ausente: {p}')
    n_npz = len(list(p.glob('*.npz')))
    print(f'    {sub}: {n_npz} arquivos .npz')

import torch
print(f'\n[4] CUDA available: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('GPU nao habilitada. Settings -> Accelerator -> T4 x1.')
print(f'    GPU: {torch.cuda.get_device_name(0)}')
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'    VRAM: {vram_gb:.1f} GB')

os.environ['TCC_RECONS_DIR'] = str(actual_root)
os.environ['TCC_ANOTADOS_DIR'] = '/kaggle/working/_dummy_anotados'
os.environ['TCC_BRAIN_CSV'] = '/kaggle/working/_dummy_brain.csv'
Path('/kaggle/working/_dummy_anotados').mkdir(exist_ok=True)
Path('/kaggle/working/_dummy_brain.csv').touch()

print('\n[5] Config do projeto:')
subprocess.run([sys.executable, '-m', 'src.config'], check=True)

RECONS_ROOT = str(actual_root)

print('\n' + '=' * 60)
print('DIAGNOSTICO OK — seguro prosseguir')
print('=' * 60)

In [ ]:
# Célula 4 — Smoke: 500 iters do Grupo B (QR).
# Gate antes do treino completo. Esperado: ~3-10 min em T4, razao final/inicial < 0.9.
import subprocess
import sys

smoke_run = '/kaggle/working/runs/group_b_smoke'

result = subprocess.run([
    sys.executable, 'scripts/smoke_train_qr.py',
    '--device', 'cuda',
    '--n-iters', '500',
    '--recons-dir', RECONS_ROOT,
    '--run-dir', smoke_run,
])

assert result.returncode == 0, (
    'SMOKE FALHOU. Veja log acima. NAO prossiga para o treino completo.'
)
print('\nSMOKE OK — seguro rodar o treino completo na proxima celula.')

In [ ]:
# Célula 5 — Treino completo: 210000 iters do Grupo B.
# T4: ~6-12h. Resume automatico em run_dir/last.pt se a sessao cair.
import subprocess
import sys

full_run = '/kaggle/working/runs/group_b_full'

result = subprocess.run([
    sys.executable, 'scripts/train_qr.py',
    '--device', 'cuda',
    '--recons-dir', RECONS_ROOT,
    '--run-dir', full_run,
    '--num-workers', '2',
])

assert result.returncode == 0, 'Treino falhou. Verifique log e last.pt.'
print('\nTreino completo concluido. Checkpoints em', full_run)

In [ ]:
# Célula 6 — Pearson sanity check (esperado r ~ 0,91 no val).
# Compara uncertainty media com error medio por slice. Se r < 0.85,
# o treino nao convergiu como o paper base; investigar antes do Grupo C.
import subprocess
import sys

result = subprocess.run([
    sys.executable, 'scripts/eval_qr_pearson.py',
    '--recons-dir', RECONS_ROOT,
    '--checkpoint', f'{full_run}/best.pt',
    '--split', 'val',
    '--device', 'cuda',
    '--threshold', '0.85',
])

assert result.returncode == 0, (
    'PEARSON FALHOU (r < 0.85). Treino do Grupo B nao replica o paper base. '
    'Investigue antes de prosseguir para o Grupo C.'
)
print('\nPEARSON OK — r >= 0.85, treino convergiu.')

In [ ]:
# Célula 7 — Empacota outputs em pasta dedicada.
import shutil
from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/tcc-mri-qr-checkpoints')
OUTPUT_DIR.mkdir(exist_ok=True)

src_dir = Path(full_run)
for fname in ('last.pt', 'best.pt', 'metrics.csv', 'config_snapshot.json'):
    src = src_dir / fname
    if src.exists():
        shutil.copy2(src, OUTPUT_DIR / fname)
        size_mb = src.stat().st_size / 1e6
        print(f'  copiado: {fname} ({size_mb:.1f} MB)')
    else:
        print(f'  AUSENTE (pulando): {fname}')

print(f'\nOutput pronto em {OUTPUT_DIR}')

In [ ]:
# Célula 8 — Publica como Kaggle dataset via API.
# Mesmo padrao da Celula 7 do kaggle_train_groupA.ipynb.
import json
import subprocess
from pathlib import Path

KAGGLE_USER = 'massanorikishi'
DATASET_SLUG = 'tcc-mri-qr-checkpoints'
DATASET_TITLE = 'TCC MRI QR Checkpoints (Grupo B, S5)'
OUTPUT_DIR = Path('/kaggle/working/tcc-mri-qr-checkpoints')

metadata = {
    'title': DATASET_TITLE,
    'id': f'{KAGGLE_USER}/{DATASET_SLUG}',
    'licenses': [{'name': 'CC0-1.0'}],
}
(OUTPUT_DIR / 'dataset-metadata.json').write_text(
    json.dumps(metadata, indent=2), encoding='utf-8'
)

print('Tentando criar dataset (primeira versao)...')
result = subprocess.run(
    ['kaggle', 'datasets', 'create', '-p', str(OUTPUT_DIR), '-r', 'zip'],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('Create falhou, tentando version:')
    print(result.stderr)
    result = subprocess.run(
        ['kaggle', 'datasets', 'version', '-p', str(OUTPUT_DIR),
         '-m', 'Update from kaggle_train_groupB notebook', '-r', 'zip'],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print('ERRO:', result.stderr)
        raise RuntimeError('Falha ao criar/atualizar dataset.')

print(f'\nDataset: https://www.kaggle.com/datasets/{KAGGLE_USER}/{DATASET_SLUG}')

## Próximos passos

Após este notebook concluir e `tcc-mri-qr-checkpoints` estar publicado:

### `notebooks/kaggle_train_groupC.ipynb` (Grupo C — QR-Lesion, contribuição original)

Mesma estrutura deste notebook, mas:

- `qr_loss` → `qr_lesion_loss`
- `loss_kwargs={'alpha': 0.10, 'lambda_lesion': 5.0}`
- `masks_dir` adicionado ao `ReconsSliceDataset` (mascaras do S3, dataset `tcc-mri-recons-varnet-brain-4x` provavelmente já inclui ou um dataset separado)

### Calibração conforme (S5.7)

Notebook separado que carrega os 3 checkpoints (A, B, C) e roda no split `cal` para computar:

- `lambda_cal` para Grupos B e C (esperado ~1,54 no B replicando paper base)
- Cobertura empirica no split `test`
- Métricas globais e por lesão (Coverage_lesion, IoU_uncertainty)

Refs: Romano et al. (2019); Angelopoulos & Bates (2023); Giannakopoulos et al. (2026, seção IV).